## 거래처 계층 구조 테이블 생성
- 오타 정제 → 입고처 제거 → Parent/Child 분류 → Duplicate 통합 → ID 재부여

In [17]:
import msoffcrypto
import io
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# 0. 파일 로드
# ─────────────────────────────────────────────
FILE_PATH = "통합170901DATA취합.xlsx"
PASSWORD  = "33"

with open(FILE_PATH, "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=PASSWORD)
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

wb = openpyxl.load_workbook(decrypted, read_only=True, data_only=True)
ws = wb["DATA"]
rows = list(ws.iter_rows(values_only=True))

columns = [
    "key", "영업담당자", "거래처명", "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g", "구단가_1",
    "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일", "제품군", "비고"
]

data = [row for row in rows[2:] if any(row)]
df_raw = pd.DataFrame(data).iloc[:, :31]
df_raw.columns = columns
df_raw = df_raw.drop(columns=["key"])
df_raw["거래처명"] = df_raw["거래처명"].astype(str).str.strip()
df_raw = df_raw[df_raw["거래처명"].notna() & (df_raw["거래처명"] != "None")].reset_index(drop=True)

print(f"원본 행 수: {len(df_raw)}")
print(f"원본 고유 거래처: {df_raw['거래처명'].nunique()}개")

원본 행 수: 5252
원본 고유 거래처: 900개


In [18]:
# ─────────────────────────────────────────────
# 1. 오타 정제
# 거래처명 전체(슬래시 앞/뒤 포함)에서 오타를 올바른 이름으로 교체
# 추가 오타 발견 시 여기에만 추가하면 됨
# ─────────────────────────────────────────────
TYPO_MAP = {
    "가지채움"      : "가치채움",
    "CJ대한퉁운"    : "CJ대한통운",
    "인펙코리아"    : "인팩코리아",
    "한국파렛트폴"  : "한국파렛트풀",
}

def fix_typo(name: str) -> str:
    """슬래시 앞/뒤 각각 오타 수정"""
    if "/" in name:
        parts = name.split("/", 1)
        parts = [TYPO_MAP.get(p.strip(), p.strip()) for p in parts]
        return "/".join(parts)
    return TYPO_MAP.get(name, name)

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(fix_typo)

fixed = (before != df_raw["거래처명"]).sum()
print(f"오타 수정된 행: {fixed}개")
print(df_raw.loc[before != df_raw["거래처명"], "거래처명"].value_counts().to_string())

오타 수정된 행: 11개
거래처명
한국파렛트풀/은해씨월드    4
가치채움/두더팩        2
한국파렛트풀/쉐프파트너    2
인팩코리아/고삼농협      2
인팩코리아/보보로지스     1


In [19]:
# ─────────────────────────────────────────────
# 2. 입고처 처리
# 슬래시 앞 토큰이 입고처면 → 앞을 제거하고 뒤(진짜 거래처)만 남김
# ex) 용인공장/한국이노팩  →  한국이노팩
# ─────────────────────────────────────────────
INGGO_SET = {
    "강원공장",
    "둔포공장",
    "용인공장",
    "서해공장",
    "아산공장",
    "싱싱패키지",
}

def strip_inggo(name: str) -> str:
    """입고처가 앞에 있으면 제거하고 뒤 이름만 반환"""
    if "/" in name:
        parent_token = name.split("/")[0].strip()
        if parent_token in INGGO_SET:
            return name.split("/", 1)[1].strip()
    return name

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(strip_inggo)

stripped = (before != df_raw["거래처명"]).sum()
print(f"입고처 제거된 행: {stripped}개")
# 변경 전후 확인
changed = before[before != df_raw["거래처명"]]
for old, new in zip(changed.values, df_raw.loc[changed.index, "거래처명"].values):
    print(f"  {old}  →  {new}")

입고처 제거된 행: 213개
  강원공장/한일익스프레스  →  한일익스프레스
  둔포공장/김완수  →  김완수
  둔포공장/김완수  →  김완수
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/청호수지  →  청호수지
  둔포공장/최동고  →  최동고
  둔포공장/최동고  →  최동고
  둔포공장/최동군  →  최동군
  둔포공장/최동군  →  최동군
  둔포공장/태성산업  →  태성산업
  싱싱패키지/건어맨  →  건어맨
  용인공장/새빛공조  →  새빛공조
  용인공장/우성AFC  →  우성AFC
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/KCP이천  →  KCP이천
  용인공장/온세물류  →  온세물류
  용인공장/온세물류  →  온세물류
  용인공장/KCP이천  →  KCP이천
  용인공장/한국이노팩  →  한국이노팩
  용인공장/한국이노팩  →  한국이노팩
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨 

In [20]:
# ─────────────────────────────────────────────
# 3. 메타 정보 추출 (정제 완료된 거래처명 기준)
# ─────────────────────────────────────────────
META_COLS = [
    "거래처명", "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일"
]
df_meta = df_raw[META_COLS].drop_duplicates(subset=["거래처명"]).set_index("거래처명")
print(f"정제 후 고유 거래처: {len(df_meta)}개")

정제 후 고유 거래처: 890개


In [21]:
# ─────────────────────────────────────────────
# 4. Master 테이블 구성
# ─────────────────────────────────────────────
all_clients = set(df_raw["거래처명"].unique())
slash_clients      = {c for c in all_clients if "/" in c}
standalone_clients = {c for c in all_clients if "/" not in c}
parent_names_from_slash = {c.split("/")[0].strip() for c in slash_clients}
auto_create_parents = parent_names_from_slash - standalone_clients

print(f"전체 고유 거래처  : {len(all_clients)}개")
print(f"  슬래시(/) 패턴  : {len(slash_clients)}개")
print(f"  단독 행         : {len(standalone_clients)}개")
print(f"  자동생성 Parent : {len(auto_create_parents)}개")

records = []

# Step A: 단독 행 (고객사)
for name in sorted(standalone_clients):
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

# Step B: 자동 생성 Parent (슬래시에만 등장)
for name in sorted(auto_create_parents):
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: None for col in META_COLS[1:]}
    })

# Step C: Child (납품처)
for name in sorted(slash_clients):
    parent_name = name.split("/")[0].strip()
    child_label = name.split("/", 1)[1].strip()
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : child_label,
        "parent_명": parent_name,
        "node_type": "납품처",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

df_master = pd.DataFrame(records).reset_index(drop=True)

# ID 부여
df_master.insert(0, "거래처_ID", range(1, len(df_master) + 1))
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

# parent_id 컬럼 위치 조정
cols = df_master.columns.tolist()
cols.remove("parent_id")
cols.insert(2, "parent_id")
df_master = df_master[cols]

# 정렬 (고객사 먼저, 납품처 뒤)
type_order = {"고객사": 0, "납품처": 1}
df_master["_sort"] = df_master["node_type"].map(type_order)
df_master = df_master.sort_values(["_sort", "거래처_ID"]).drop(columns=["_sort"]).reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

전체 고유 거래처  : 890개
  슬래시(/) 패턴  : 252개
  단독 행         : 638개
  자동생성 Parent : 29개


In [22]:
# ─────────────────────────────────────────────
# 5. Duplicate 통합
# 같은 거래처명이 고객사 + 납품처 둘 다 있는 경우:
#   → 고객사 ID가 진짜 parent
#   → 납품처 행 삭제
#   → 납품처 ID를 가리키던 child들의 parent_id를 고객사 ID로 재연결
#
# ※ 단, 입고처(INGGO_SET)가 parent였던 케이스는 이미 Step2에서 제거됨
#   → 여기서 걸리는 건 진짜 거래처가 다른 거래처의 납품처로도 등록된 경우만
# ─────────────────────────────────────────────
고객사_names = set(df_master[df_master["node_type"] == "고객사"]["거래처명"])
납품처_names = set(df_master[df_master["node_type"] == "납품처"]["거래처명"])
duplicates = 고객사_names & 납품처_names

print(f"Duplicate 거래처: {len(duplicates)}개")
for name in sorted(duplicates):
    print(f"  • {name}")

for name in duplicates:
    real_id = df_master.loc[
        (df_master["거래처명"] == name) & (df_master["node_type"] == "고객사"),
        "거래처_ID"
    ].values[0]

    fake_id = df_master.loc[
        (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"),
        "거래처_ID"
    ].values[0]

    # fake를 가리키던 child → real로 재연결
    mask = df_master["parent_id"] == fake_id
    df_master.loc[mask, "parent_id"] = real_id
    df_master.loc[mask, "parent_명"] = name

    # 납품처 행 삭제
    df_master = df_master[
        ~((df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"))
    ]

# ID 재부여
df_master = df_master.reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

print(f"\n정리 후 전체: {len(df_master)}개")
print(f"  고객사: {(df_master['node_type']=='고객사').sum()}개")
print(f"  납품처: {(df_master['node_type']=='납품처').sum()}개")

Duplicate 거래처: 16개
  • 건어맨
  • 남선푸드
  • 동아에스티
  • 디베랴
  • 로지스프로
  • 메이플델리
  • 명정보기술
  • 바이오플레이
  • 아이탐푸드
  • 우진수산
  • 유한산업
  • 제이월드텍
  • 지웅
  • 진원
  • 태진패키지
  • 팀프레시

정리 후 전체: 903개
  고객사: 667개
  납품처: 236개


In [23]:
# ─────────────────────────────────────────────
# 6. 검증
# ─────────────────────────────────────────────
print("【Parent → Child 샘플】")
for p_name in ["청호나이스", "CJ대한통운", "동원홈푸드", "한국파렛트풀", "한국이노팩", "컬리"]:
    if p_name not in name_to_id:
        print(f"  {p_name} → 없음")
        continue
    pid  = name_to_id[p_name]
    row  = df_master[df_master["거래처_ID"] == pid].iloc[0]
    kids = df_master[df_master["parent_id"] == pid][["거래처_ID","거래처명"]].values
    print(f"  [{pid}] {p_name} ({row['node_type']})")
    for kid_id, kid_name in kids[:5]:
        print(f"       └─ [{int(kid_id)}] {kid_name}")
    if len(kids) > 5:
        print(f"       └─ ... 외 {len(kids)-5}개")

# parent_id가 존재하지 않는 ID를 가리키는 행 확인 (정합성 체크)
valid_ids = set(df_master["거래처_ID"])
broken = df_master[df_master["parent_id"].notna() & ~df_master["parent_id"].isin(valid_ids)]
print(f"\n깨진 parent_id 참조: {len(broken)}개")
if len(broken) > 0:
    print(broken[["거래처_ID","거래처명","parent_id","parent_명"]])

【Parent → Child 샘플】
  [517] 청호나이스 (고객사)
       └─ [842] 미래씨엔에이치
       └─ [843] 에스엠나이스
       └─ [844] 협신산업
  [641] CJ대한통운 (고객사)
       └─ [673] 가미락
       └─ [674] 강원도옥수수총각
       └─ [675] 그림엘푸드
       └─ [676] 남천영농조합법인
       └─ [677] 도당
       └─ ... 외 21개
  [643] 동원홈푸드 (고객사)
       └─ [721] 가산공장
       └─ [722] 금천미트(대전)
       └─ [723] 대전센터
       └─ [724] 시화센터
       └─ [725] 장호원
       └─ ... 외 1개
  [665] 한국파렛트풀 (고객사)
       └─ [877] 대상
       └─ [878] 더다냉동물류
       └─ [879] 더유리빙
       └─ [880] 동원로엑스
       └─ [881] 두레냉동
       └─ ... 외 16개
  [603] 한국이노팩 (고객사)
       └─ [874] 이푸드부산
       └─ [875] 이푸드평택
       └─ [876] 현대그린푸드
  [660] 컬리 (고객사)
       └─ [845] 남양주
       └─ [846] 송파

깨진 parent_id 참조: 0개


In [24]:
df_master[df_master['parent_명'] == '이천']

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일


In [25]:
df_master[df_master['거래처명'].duplicated(keep = False)]

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
699,700,덕평,8.0,GS네트웍스,납품처,NaN,NaN,NaN,NaN,NaN,경기 이천시 호법면 덕평로 566,NaN,NaN,NaN,최지훈,NaN,010-6554-1100,None,None
703,704,송파,642.0,GS리테일,납품처,허연수,755-85-00261,서울 송파구 장지동 674 일원 동남권유통단지 C동 B층,소매,통신판매,서울 송파구 송파대로55 서울복합물류단지 C동 지하1층창고,한민성 대리,777hanms@gsretail.com,010-7212-7395,김선모,leesen@gsretail.com,010-4590-9627,None,30
708,709,이천,10.0,HM솔루션,납품처,곽희창,218-17-62513,경기도 평택시 매봉산3길 5-3(비전동),"제조업, 도매 및 소매업","아이스팩, 전자상거래",None,NaN,NaN,NaN,김용덕 부장,NaN,010-2655-9758,None,None
715,716,이천,58.0,구르메,납품처,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,010-2558-1263,None,None
718,719,김포,102.0,대명,납품처,윤효정,492-24-00637,강원도 횡성군 우천면 전재로 129,"제조업, 도매 및 소매업",스티로폼박스,None,NaN,NaN,NaN,NaN,NaN,010-4339-0009,None,None
726,727,광주,167.0,로지텍,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
732,733,인천,257.0,서울우유몰,납품처,정수경,210-11-17685,인천광역시 부평구 십정동 98-1,도소매,"유제품,생수",경기도 안성시 신두만곡로 181,NaN,cheesemart@hanmail.net,NaN,NaN,NaN,010-3938-0112,None,None
771,772,광주,343.0,아침애과일,납품처,NaN,129-36-02161,성남시 수정구 탄리로 125번길2,소매업,전자상거래업,경기도 안성시 신두만곡로 181,윤찬수,chansoo.yoon85@gmail.com,070-4079-1313,NaN,NaN,010-2426-3250,2.5톤 9만원,30
772,773,인천,345.0,아풍오닉스,납품처,NaN,124-87-19131,경기도 화성시 장안면 노진길 140-63,제조업,자외선살균소독기,경기도 안성시 신두만곡로 181,장미,ahpoongo@hanmail.net,010-3898-5338,장미,ahpoongo@hanmail.net,010-2880-1471,None,30
790,791,안성,403.0,용마로지스,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,이원표 과장님,NaN,010-9492-6255,None,None


In [8]:
# ─────────────────────────────────────────────
# 7. Detail 테이블 (FK 연결)
# ─────────────────────────────────────────────

# df_raw 거래처명도 동일하게 오타/입고처 정제 반영됨 (이미 위에서 처리)
# child는 child_label(슬래시 뒤)로 저장했으므로 거래처명 매핑 필요

# slash_clients는 원본 슬래시 포함 이름 → child_label로 변환해서 매핑
slash_label_map = {}
for name in slash_clients:
    parent_token = name.split("/")[0].strip()
    child_label  = name.split("/", 1)[1].strip()
    # 입고처였으면 df_raw에서 이미 child_label로 바뀌어있음
    slash_label_map[name] = child_label

# df_raw의 거래처명은 정제 완료 상태, name_to_id로 바로 연결
df_raw["거래처_ID"] = df_raw["거래처명"].map(name_to_id)

DETAIL_COLS = [
    "거래처_ID", "거래처명", "영업담당자",
    "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g",
    "구단가_1", "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "제품군", "비고"
]
df_detail = df_raw[DETAIL_COLS].reset_index(drop=True)
df_detail.insert(0, "거래내역_ID", range(1, len(df_detail) + 1))

for col in ["단가", "합의중량_g", "구단가_1", "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7"]:
    df_detail[col] = pd.to_numeric(df_detail[col], errors="coerce")

print(f"Detail 행 수: {len(df_detail)}")
print(f"거래처_ID 매핑 실패: {df_detail['거래처_ID'].isna().sum()}건")

Detail 행 수: 5252
거래처_ID 매핑 실패: 1073건


In [9]:
# ─────────────────────────────────────────────
# 8. Excel 저장
# ─────────────────────────────────────────────
OUTPUT_PATH = "SQL_계층구조_거래처DB.xlsx"

COLORS = {
    "header" : "1B2A3B",
    "고객사_bg" : "D6EAF8", "고객사_font": "1A5276",
    "납품처_bg" : "D5F5E3", "납품처_font": "1E8449",
    "alt_row"  : "F4F6F7",
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_master.to_excel(writer, sheet_name="Master_거래처", index=False)
    df_detail.to_excel(writer, sheet_name="Detail_거래내역", index=False)

    for sheet_name, df in [("Master_거래처", df_master), ("Detail_거래내역", df_detail)]:
        ws = writer.sheets[sheet_name]

        # 헤더
        h_fill = PatternFill("solid", start_color=COLORS["header"], end_color=COLORS["header"])
        h_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        for cell in ws[1]:
            cell.fill = h_fill
            cell.font = h_font
            cell.alignment = Alignment(horizontal="center", vertical="center")

        # 행 색상 (Master만)
        if sheet_name == "Master_거래처":
            nt_col = df_master.columns.get_loc("node_type") + 1
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                nt = ws.cell(row=row_idx, column=nt_col).value
                bg   = COLORS.get(f"{nt}_bg",   "FFFFFF")
                fc   = COLORS.get(f"{nt}_font", "000000")
                bold = (nt == "고객사")
                fill = PatternFill("solid", start_color=bg, end_color=bg)
                font = Font(color=fc, bold=bold, name="Arial", size=9)
                for cell in row:
                    cell.fill = fill
                    cell.font = font
        else:
            alt = PatternFill("solid", start_color=COLORS["alt_row"], end_color=COLORS["alt_row"])
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for cell in row:
                    cell.font = Font(name="Arial", size=9)
                    if row_idx % 2 == 0:
                        cell.fill = alt

        # 열 너비
        for col in ws.columns:
            w = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 45)

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print(f"✅ 저장 완료: {OUTPUT_PATH}")

PermissionError: [Errno 13] Permission denied: 'SQL_계층구조_거래처DB.xlsx'